# ML.ENERGY V3 Phase 1.5 — 完整汇总与工作量字段审计

## tl;dr

**判定 B：有条件可继续。** 694 条文本 LLM inference 汇总记录内部一致、稳定子集与 Phase 1 完全一致；但正式建模前必须独立重构并冻结事前输入 token 工作量/分布。本文不训练模型，只复现审计证据。

## Context & Methods

- 范围仅含 GPQA、LM Arena Chat、Sourcegraph FIM 三类文本 LLM inference。
- 指标是稳态窗口内所有 GPU 的聚合 GPU 侧口径。
- 笔记本只读取 Git 中的审计 JSON，不读取 gated parquet、Prometheus 或 timeline。
- 输入变量按执行前已知/可推导与执行后泄漏严格分开。

In [1]:
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'step1' / 'analysis').exists():
    ROOT = ROOT.parent.parent

def load(name):
    return json.loads((ROOT / 'step1' / 'analysis' / name).read_text(encoding='utf-8'))

full = load('full_parquet_audit.json')
comparison = load('leaderboard_comparison.json')
selected = load('selected_runs.json')
workload = load('workload_field_audit.json')

assert full['row_count'] == 694
assert full['stability']['stable'] == 565
assert comparison['multiset_difference'] == 0
assert workload['decision']['code'] == 'B'
assert workload['scope']['timeline_payloads_retained'] == 0
print('Reviewed outputs loaded and headline assertions passed.')

Reviewed outputs loaded and headline assertions passed.


## Data

In [2]:
summary = f'''| 指标 | 值 |
|---|---:|
| parquet 全部 LLM+MLLM 行 | {full['source_population']['parquet_all_llm_and_mllm_rows']} |
| 排除 MLLM 行 | {full['source_population']['excluded_mllm_rows']} |
| 文本 LLM 行 | {full['row_count']} |
| 稳定 / 非稳定 | {full['stability']['stable']} / {full['stability']['unstable']} |
| 模型数 | {full['cross_coverage']['models']} |
| task×GPU×model 单元 | {full['cross_coverage']['observed_task_gpu_model_cells']} |
| 代表性 metadata | {selected['selected_count']} |
| 保留 timeline | {workload['scope']['timeline_payloads_retained']} |
'''
display(Markdown(summary))

| 指标 | 值 |
|---|---:|
| parquet 全部 LLM+MLLM 行 | 1138 |
| 排除 MLLM 行 | 444 |
| 文本 LLM 行 | 694 |
| 稳定 / 非稳定 | 565 / 129 |
| 模型数 | 27 |
| task×GPU×model 单元 | 58 |
| 代表性 metadata | 11 |
| 保留 timeline | 0 |


In [3]:
import matplotlib.pyplot as plt

tasks = ['gpqa', 'lm-arena-chat', 'sourcegraph-fim']
stable = [full['stability']['by_task'][t]['stable'] for t in tasks]
unstable = [full['stability']['by_task'][t]['unstable'] for t in tasks]

fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.bar(tasks, stable, color='#2176AE', label='Stable')
ax.bar(tasks, unstable, bottom=stable, color='#F4A261', label='Unstable')
ax.set_ylabel('Run rows')
ax.set_title('Text-LLM stability by benchmark task')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

C:\Users\CPECC\AppData\Local\Temp\ipykernel_37104\631395043.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Results

In [4]:
reason_lines = '\n'.join(
    f"- `{reason}`: {count}" for reason, count in full['stability']['unstable_reason_categories'].items()
)
display(Markdown('### 非稳定原因\n\n' + reason_lines))

checks = full['consistency']
check_table = '| 校验 | 检查行 | 失败行 | 最大相对误差 |\n|---|---:|---:|---:|\n'
for name, result in checks.items():
    check_table += f"| `{name}` | {result['checked']} | {result['failed']} | {result.get('maximum_relative_error', '—')} |\n"
display(Markdown('### 一致性校验\n\n' + check_table))

### 非稳定原因

- `cascade_from_unstable_batch`: 2
- `low_batch_utilization`: 122
- `short_steady_state`: 5

### 一致性校验

| 校验 | 检查行 | 失败行 | 最大相对误差 |
|---|---:|---:|---:|
| `actual_average_output_length` | 694 | 0 | 0.0 |
| `energy_power_duration` | 694 | 0 | 0.0 |
| `itl_percentile_order` | 694 | 0 | — |
| `power_from_energy_intensity_and_throughput` | 694 | 0 | 2.483118554175566e-16 |
| `request_energy_from_actual_output` | 694 | 0 | 0.0 |


In [5]:
header = '| task | GPU | architecture | scale | GPUs | max_num_seqs | TP/EP/DP |\n|---|---|---|---|---:|---:|---|\n'
body = ''
for run in selected['runs']:
    body += (
        f"| {run['task']} | {run['gpu_model']} | {run['architecture']} | {run['model_scale']} | "
        f"{run['num_gpus']} | {run['max_num_seqs']} | "
        f"{run['tensor_parallel']}/{run['expert_parallel']}/{run['data_parallel']} |\n"
    )
display(Markdown('### 代表性 run 覆盖\n\n' + header + body))

### 代表性 run 覆盖

| task | GPU | architecture | scale | GPUs | max_num_seqs | TP/EP/DP |
|---|---|---|---|---:|---:|---|
| gpqa | B200 | Dense Transformer | small | 1 | 128 | 1/1/1 |
| gpqa | H100 | MoE | large | 2 | 1024 | 2/1/1 |
| lm-arena-chat | B200 | MoE | large | 2 | 16 | 1/2/1 |
| lm-arena-chat | H100 | Dense Transformer | medium | 4 | 1024 | 4/1/1 |
| sourcegraph-fim | B200 | MoE | large | 8 | 1024 | 1/8/1 |
| sourcegraph-fim | H100 | MoE | medium | 1 | 16 | 1/1/1 |
| gpqa | B200 | MoE | large | 4 | 128 | 1/4/1 |
| gpqa | B200 | Mamba-Transformer Hybrid | small | 1 | 128 | 1/1/1 |
| lm-arena-chat | B200 | Dense Transformer | large | 8 | 1024 | 8/1/1 |
| sourcegraph-fim | B200 | MoE | large | 4 | 128 | 1/4/4 |
| sourcegraph-fim | B200 | MoE | large | 8 | 1024 | 1/8/8 |


In [6]:
from collections import Counter

categories = Counter(item['category'] for item in workload['field_audit'].values())
display(Markdown('### 结果头部字段分类\n\n' + '\n'.join(f'- `{k}`: {v}' for k, v in sorted(categories.items()))))

matrix = '| task | unique prompts | repeats in sample | max output cap | endpoint | source check |\n|---|---:|---|---:|---|---|\n'
for task, spec in workload['task_matrix'].items():
    matrix += (
        f"| {task} | {spec['num_unique_prompts']} | {spec['observed_repeat_counts']} | "
        f"{spec['max_output_tokens']} | {spec['endpoint_type']} | {workload['cross_checks'][task]['status']} |\n"
    )
display(Markdown('### 工作量配置交叉核对\n\n' + matrix))

### 结果头部字段分类

- `diagnostic`: 2
- `diagnostic_outcome`: 8
- `identifier_risk`: 1
- `out_of_scope`: 16
- `post_run_leakage`: 10
- `pre_derivable`: 1
- `pre_known`: 11
- `target`: 10

### 工作量配置交叉核对

| task | unique prompts | repeats in sample | max output cap | endpoint | source check |
|---|---:|---|---:|---|---|
| gpqa | 198 | [1, 6] | 32768 | openai-chat | pass |
| lm-arena-chat | 1024 | [1, 2] | 4096 | openai-chat | pass |
| sourcegraph-fim | 1024 | [1, 3, 4] | 2048 | openai | pass |


## Takeaways

1. `E=P×T` 等四组恒等式在 694 行上全部通过，稳定子集与 Phase 1 投影差异为 0。
2. `num_request_repeats` 是计划工作量变量，不是统计学上的独立重复；全体 seed 固定为 48105。
3. 允许进入 `X_pre` 的是任务、模型物理元数据、GPU/卡数/并行与运行前流量配置。
4. 实际 input/output tokens、吞吐、时长、ITL、平均 batch、稳定性与能源测量禁止作为输入。
5. 最大缺口是未持久化的事前输入 token 总量/分布；先补齐该表，再讨论正式模型。

**最终判定：B（有条件可继续）；Phase 1.5 到此停止。**